In [2]:
import threading
import queue
import time
import random

# 1. Keep an infinite capacity queue to observe the build-up
backlog_queue = queue.Queue()
stop_pipeline = threading.Event()

def fast_producer():
    """
    PRODUCER: Generates data very quickly (every 0.05 seconds)
    """
    print(f"[{threading.current_thread().name}] Fast producer started.")
    reading_id = 1

    while not stop_pipeline.is_set():
        mock_data = {"id": reading_id, "val": round(random.uniform(10.0, 90.0), 2)}

        backlog_queue.put(mock_data)
        print(f"[{threading.current_thread().name}] Sent packet #{reading_id} -> Queue Size: {backlog_queue.qsize()}")

        reading_id += 1
        time.sleep(0.05)  # Super fast output! (20 packets per second)

    print(f"[{threading.current_thread().name}] Producer stopped.")


def slow_consumer():
    """
    CONSUMER: Processes data much slower (every 0.3 seconds)
    """
    print(f"[{threading.current_thread().name}] Slow consumer started.")

    while not stop_pipeline.is_set() or not backlog_queue.empty():
        try:
            data_packet = backlog_queue.get(timeout=0.1)

            # Print what packet the consumer is currently working on
            print(f"    [{threading.current_thread().name}] Consumer caught packet #{data_packet['id']}... Processing slowly...")
            time.sleep(0.3)  # Slow processing speed! (Only ~3 packets per second)

            backlog_queue.task_done()

        except queue.Empty:
            continue

    print(f"[{threading.current_thread().name}] Consumer stopped.")

# --- Execution ---
prod_thread = threading.Thread(target=fast_producer, name="Fast_Producer")
cons_thread = threading.Thread(target=slow_consumer, name="Slow_Consumer")

prod_thread.start()
cons_thread.start()

# Let the imbalance run for 1 second to watch the queue accumulate
time.sleep(1.0)

print("\n[Main] Stopping pipeline production...")
stop_pipeline.set()

prod_thread.join()
cons_thread.join()

print("[Main] Simulation finished.")

[Fast_Producer] Fast producer started.
[Fast_Producer] Sent packet #1 -> Queue Size: 1
[Slow_Consumer] Slow consumer started.
    [Slow_Consumer] Consumer caught packet #1... Processing slowly...
[Fast_Producer] Sent packet #2 -> Queue Size: 1
[Fast_Producer] Sent packet #3 -> Queue Size: 2
[Fast_Producer] Sent packet #4 -> Queue Size: 3
[Fast_Producer] Sent packet #5 -> Queue Size: 4
[Fast_Producer] Sent packet #6 -> Queue Size: 5
    [Slow_Consumer] Consumer caught packet #2... Processing slowly...
[Fast_Producer] Sent packet #7 -> Queue Size: 5
[Fast_Producer] Sent packet #8 -> Queue Size: 6
[Fast_Producer] Sent packet #9 -> Queue Size: 7
[Fast_Producer] Sent packet #10 -> Queue Size: 8
[Fast_Producer] Sent packet #11 -> Queue Size: 9
[Fast_Producer] Sent packet #12 -> Queue Size: 10
    [Slow_Consumer] Consumer caught packet #3... Processing slowly...
[Fast_Producer] Sent packet #13 -> Queue Size: 10
[Fast_Producer] Sent packet #14 -> Queue Size: 11
[Fast_Producer] Sent packet #15 